Name : Bhushan Mandekar

Batch : 14


# Insurance Claim Processing Agent using LangGraph

## Objective
Build an AI-powered insurance claim workflow using LangGraph,
Gemini 2.5 Flash, and Streamlit.

## Agents
1. Document Verification Agent
2. Eligibility Check Agent
3. Fraud Detection Agent
4. Claim Summary Agent
5. Human Approval Agent

The document, eligibility, and fraud checks run in parallel.
Conditional edges route claims to approval, rejection, or human review.


Load configuration

In [1]:

from dotenv import load_dotenv
import os

load_dotenv()

print("Gemini model:", os.getenv("GEMINI_MODEL", "gemini-3.6-flash"))
print("API key configured:", bool(os.getenv("GOOGLE_API_KEY")))


Gemini model: gemini-3.6-flash
API key configured: True


Import the LangGraph workflow

In [2]:

from workflow import (
    claim_graph,
    process_claim,
    REQUIRED_DOCUMENTS
)

print("LangGraph workflow loaded successfully.")
print(claim_graph)


LangGraph workflow loaded successfully.


Define four test scenarios

In [3]:

scenarios = [
    {
        "claim_id": "TEST-001",
        "claimant_name": "Meera Patil",
        "claim_type": "vehicle",
        "claim_amount": 42000,
        "incident_date": "2026-09-15",
        "policy_start": "2026-01-01",
        "policy_end": "2026-12-31",
        "policy_status": "active",
        "description": "Vehicle damage with repair estimate.",
        "submitted_documents": REQUIRED_DOCUMENTS["vehicle"],
    },
    {
        "claim_id": "TEST-002",
        "claimant_name": "Rohan Das",
        "claim_type": "health",
        "claim_amount": 32000,
        "incident_date": "2026-09-20",
        "policy_start": "2026-01-01",
        "policy_end": "2026-12-31",
        "policy_status": "active",
        "description": "Hospitalization claim.",
        "submitted_documents": ["claim_form", "policy_copy"],
    },
    {
        "claim_id": "TEST-003",
        "claimant_name": "Ananya Rao",
        "claim_type": "property",
        "claim_amount": 350000,
        "incident_date": "2026-08-10",
        "policy_start": "2026-01-01",
        "policy_end": "2026-12-31",
        "policy_status": "active",
        "description": "Property damage after a storm.",
        "submitted_documents": REQUIRED_DOCUMENTS["property"],
    },
    {
        "claim_id": "TEST-004",
        "claimant_name": "Kabir Singh",
        "claim_type": "travel",
        "claim_amount": 18000,
        "incident_date": "2026-09-01",
        "policy_start": "2025-01-01",
        "policy_end": "2025-12-31",
        "policy_status": "expired",
        "description": "Travel expense claim.",
        "submitted_documents": REQUIRED_DOCUMENTS["travel"],
    },
]

print("Number of test scenarios:", len(scenarios))


Number of test scenarios: 4


Run the test cases

In [4]:

results = []

for claim in scenarios:
    result = process_claim(claim)
    results.append(result)

    print("=" * 60)
    print("Claim ID:", result["claim_id"])
    print("Decision:", result["decision"])
    print("Reason:", result["routing_reason"])
    print("Document status:", result["document_result"]["status"])
    print("Eligibility status:", result["eligibility_result"]["status"])
    print("Fraud status:", result["fraud_result"]["status"])
    print("Summary:")
    print(result["claim_summary"])


d:\AIML\VS code project deployment\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim ID: TEST-001
Decision: AUTO_APPROVED
Reason: Required documents and eligibility checks passed; no risk flags were detected.
Document status: PASS
Eligibility status: PASS
Fraud status: CLEAR
Summary:
[{'type': 'text', 'text': 'Here is the insurance claim summary:\n\n* **Claim Overview:** Claim ID TEST-001 is a vehicle insurance claim filed by claimant Meera Patil for an amount of INR 42,000.00 regarding an incident on 2026-09-15.\n* **Eligibility Status:** The eligibility status is PASS, as the policy is active and the incident date falls within coverage dates.\n* **Document Status:** The document status is PASS, with no missing documents reported.\n* **Fraud Screening:** The fraud screen status is CLEAR, and there are zero risk flags present.\n* **Important Note:** Fraud flags serve only as indicators and do not constitute absolute proof; please note that a final claim decision is not being made in this summary.', 'extras': {'signature': 'EugaCuUaAWkUfRMpish0WO8EHCkp4vSS6buX1Ix1

d:\AIML\VS code project deployment\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim ID: TEST-002
Decision: REJECTED
Reason: Missing documents: ['hospital_bill', 'discharge_summary', 'identity_proof']
Document status: FAIL
Eligibility status: PASS
Fraud status: CLEAR
Summary:
[{'type': 'text', 'text': "Here is the concise summary for the insurance claim:\n\n* **Claim Details:** Claim ID TEST-002 was filed by claimant Rohan Das for a health insurance claim of INR 32,000.00 relating to an incident date of 2026-09-20.\n* **Eligibility Status:** The eligibility status is PASS, as the policy is active and the incident date is within the coverage dates.\n* **Document Status:** The document status is FAIL due to missing required documents, specifically: 'hospital_bill', 'discharge_summary', and 'identity_proof'.\n* **Fraud & Risk Screen:** The fraud screen is CLEAR, and there are currently no risk flags present.\n* **Important Note:** Fraud flags serve only as risk indicators and do not constitute definitive proof of fraud; no final claim decision has been made.", 'extr

d:\AIML\VS code project deployment\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim ID: TEST-003
Decision: PENDING_HUMAN_REVIEW
Reason: High-value claim: amount is at least INR 200000.
Document status: PASS
Eligibility status: PASS
Fraud status: FLAG
Summary:
[{'type': 'text', 'text': 'Here is a summary of the insurance claim:\n\n* **Claim Overview:** Claim ID TEST-003 is a property claim submitted by Ananya Rao for the amount of INR 350,000.00, regarding an incident date of 2026-08-10.\n* **Document Status:** Document status is recorded as PASS, with no missing documents.\n* **Eligibility Status:** Eligibility status is recorded as PASS because the policy is active and the incident date is within coverage dates.\n* **Fraud Screening:** The fraud screen status is set to FLAG due to a risk flag indicating a high-value claim (amount is at least INR 200,000.00).\n* **Fraud Flag Explanation:** Please note that fraud flags serve only as risk indicators for further review and do not constitute proof of fraud. \n* **Final Status:** A final claim decision has not been m

d:\AIML\VS code project deployment\myenv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim ID: TEST-004
Decision: REJECTED
Reason: Policy status is not active.; Incident occurred after policy expiry.
Document status: PASS
Eligibility status: FAIL
Fraud status: CLEAR
Summary:
[{'type': 'text', 'text': 'Here is the claim summary based on the provided data:\n\n* **Claim Overview:** Claim ID TEST-004 is a travel claim submitted by Kabir Singh for the amount of INR 18,000.00 for an incident on 2026-09-01.\n* **Document Status:** Document status is PASS, with no missing documents reported.\n* **Eligibility Assessment:** The eligibility status is FAIL because the policy status is not active and the incident occurred after policy expiry.\n* **Fraud & Risk Screening:** The fraud screen is CLEAR and there are no risk flags recorded.\n* **Fraud Flag Clarification:** Please note that fraud flags are strictly indicators for evaluation, not definitive proof of fraud.\n* **Claim Decision:** No final claim decision is made in this summary.', 'extras': {'signature': 'EqEhCp4hAWkUfRPNET

Check expected outcomes

In [5]:

expected_decisions = [
    "AUTO_APPROVED",
    "REJECTED",
    "PENDING_HUMAN_REVIEW",
    "REJECTED",
]

for claim, result, expected in zip(
    scenarios, results, expected_decisions
):
    actual = result["decision"]
    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{claim['claim_id']}: {status} "
        f"| Expected: {expected} | Actual: {actual}"
    )


TEST-001: PASS | Expected: AUTO_APPROVED | Actual: AUTO_APPROVED
TEST-002: PASS | Expected: REJECTED | Actual: REJECTED
TEST-003: PASS | Expected: PENDING_HUMAN_REVIEW | Actual: PENDING_HUMAN_REVIEW
TEST-004: PASS | Expected: REJECTED | Actual: REJECTED
